# 02 EDA

Goal: understand the cleaned headline data before any sentiment scoring or modeling. This notebook focuses on data quality, headline volume drift, return targets, macro context, and regime-level patterns.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".cache" / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / ".cache"))

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "sp500_headlines_2008_2024.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

In [ ]:
raw = pd.read_csv(RAW_PATH, parse_dates=["Date"])
clean = pd.read_csv(PROCESSED_DIR / "headlines_clean.csv", parse_dates=["Date"])
daily = pd.read_csv(PROCESSED_DIR / "daily_with_macro.csv", parse_dates=["Date"])
headlines_by_year = pd.read_csv(PROCESSED_DIR / "eda_headlines_by_year.csv")
duplicates_by_year = pd.read_csv(PROCESSED_DIR / "eda_duplicates_by_year.csv")
regime_summary = pd.read_csv(PROCESSED_DIR / "eda_regime_summary.csv")
macro_coverage = pd.read_csv(PROCESSED_DIR / "eda_macro_coverage.csv")
return_extremes = pd.read_csv(PROCESSED_DIR / "eda_return_extremes.csv", parse_dates=["Date"])
possible_off_topic = pd.read_csv(PROCESSED_DIR / "eda_possible_off_topic_sample.csv", parse_dates=["Date"])

daily.head()

## Data Quality Checks

In [ ]:
quality_summary = pd.DataFrame({
    "metric": [
        "raw rows",
        "clean rows",
        "exact duplicates removed",
        "raw missing values",
        "unique trading dates",
        "prepared daily rows",
    ],
    "value": [
        len(raw),
        len(clean),
        raw.duplicated().sum(),
        int(raw.isna().sum().sum()),
        raw["Date"].nunique(),
        len(daily),
    ],
})
quality_summary

In [ ]:
duplicates_by_year.sort_values("duplicate_rows_removed", ascending=False).head(10)

## Headline Volume Drift

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.barplot(data=headlines_by_year, x="year", y="headline_rows", ax=axes[0], color="#2f6f8f")
axes[0].set_title("Headline Volume by Year")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Headlines after duplicate removal")
axes[0].tick_params(axis="x", rotation=45)

sns.lineplot(data=headlines_by_year, x="year", y="avg_headlines_per_day", marker="o", ax=axes[1], color="#7a4f9a")
axes[1].set_title("Average Headlines per Trading Day")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Headlines per trading day")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "headline_volume_and_density.png", dpi=200)
plt.show()

## Price and Return Targets

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(daily["Date"], daily["CP"], color="#263238")
axes[0].set_title("S&P 500 Closing Price")
axes[0].set_ylabel("Close")
axes[1].plot(daily["Date"], daily["return_next_day"], color="#8f3f2f", linewidth=0.8)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Next-Day Return Target")
axes[1].set_ylabel("Return")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "sp500_price_and_returns.png", dpi=200)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(daily["return_next_day"], bins=80, kde=True, ax=ax, color="#5b7894")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Distribution of Next-Day Returns")
ax.set_xlabel("Next-day return")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "next_day_return_distribution.png", dpi=200)
plt.show()

In [ ]:
return_extremes[["Date", "return_next_day", "headline_count", "regime", "vix"]].sort_values(
    "return_next_day", key=lambda s: s.abs(), ascending=False
).head(10)

## Macro and Regime Context

In [ ]:
macro_coverage

In [ ]:
regime_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=daily, x="regime", y="return_next_day", ax=ax)
ax.set_title("Next-Day Returns by Regime")
ax.set_xlabel("")
ax.set_ylabel("Next-day return")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "next_day_returns_by_regime.png", dpi=200)
plt.show()

## Possible Off-Topic Headline Check

This is only a quick quality audit. It flags headlines that do not contain a small set of obvious finance or market terms, so the sample should be reviewed manually rather than removed automatically.

In [ ]:
possible_off_topic[["Date", "Title", "CP"]].head(20)